In [1]:
# Importar librerias
import pandas as pd

from funciones import *

In [2]:
# Generar lista de tickers del Fichero constituents del S&P 500
data_folder = "../data"
df_tickers = pd.read_csv(f"{data_folder}/constituents.csv")

# Modificar "BRK.B" a "BRK-B" y "BF.B" a "BF-B" para evitar problemas con yfinance
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BRK.B", "BRK-B")
df_tickers["Symbol"] = df_tickers["Symbol"].replace("BF.B", "BF-B")

tickers_list = df_tickers["Symbol"].tolist()
len(tickers_list)

503

In [3]:
# Seleccionar y renombrar columnas en df_tickers
df_tickers = df_tickers[["Symbol", "GICS Sector","GICS Sub-Industry", "Date added"]]
df_tickers.rename(columns={
    "Symbol": "Ticker",
    "GICS Sector": "Sector",
    "GICS Sub-Industry": "SubIndustry", 
    "Date added": "DateAdded"
    }, inplace=True)

In [4]:
df_tickers.head()

,Ticker,Sector,SubIndustry,DateAdded
0,MMM,Industrials,Industrial Conglomerates,1957-03-04
1,AOS,Industrials,Building Products,2017-07-26
2,ABT,Health Care,Health Care Equipment,1957-03-04
3,ABBV,Health Care,Biotechnology,2012-12-31
4,ACN,Information Technology,IT Consulting & Other Services,2011-07-06


In [ ]:
df_prices = extraer_precios(tickers_list)

In [6]:
df_index = extraer_precios(['SPY'])
df_betas = calcular_betas(df_prices, df_index)
df_betas.head()

,Fecha,Ticker,Beta
0,2022-06-06,AAPL,NaN
1,2022-06-13,AAPL,NaN
2,2022-06-20,AAPL,NaN
3,2022-06-27,AAPL,NaN
4,2022-07-04,AAPL,NaN


In [7]:
df_betas.tail()

,Fecha,Ticker,Beta
413,2026-05-04,MSFT,1.0795
414,2026-05-11,MSFT,1.1259
415,2026-05-18,MSFT,1.1822
416,2026-05-25,MSFT,1.2199
417,2026-06-01,MSFT,1.2897


In [8]:
df_prices.head()

,Fecha,Close,Ticker
0,2022-06-06,134.431396,AAPL
1,2022-06-13,128.971008,AAPL
2,2022-06-20,138.872253,AAPL
3,2022-06-27,136.195969,AAPL
4,2022-07-04,144.146393,AAPL


In [9]:
df_index.head()

,Fecha,Close,Capital Gains,Ticker
0,2022-06-06,368.857452,0.0,SPY
1,2022-06-13,346.203644,0.0,SPY
2,2022-06-20,370.716919,0.0,SPY
3,2022-06-27,362.315735,0.0,SPY
4,2022-07-04,369.376953,0.0,SPY


In [ ]:
df_fundamentals = extraer_datos_fundamentales(tickers_list)

In [11]:
df_fundamentals.head()

,Fecha,Total Revenue,Gross Profit,Operating Income,Net Income,EBITDA,Basic Average Shares,Cash And Cash Equivalents,Current Debt,Long Term Debt,Total Debt,Stockholders Equity,Total Assets,Current Assets,Current Liabilities,Ticker
0,2025-09-30,4.161610e+11,1.952010e+11,1.330500e+11,1.120100e+11,1.447480e+11,1.494850e+10,3.593400e+10,2.032900e+10,7.832800e+10,9.865700e+10,7.373300e+10,3.592410e+11,1.479570e+11,1.656310e+11,AAPL
1,2024-09-30,3.910350e+11,1.806830e+11,1.232160e+11,9.373600e+10,1.346610e+11,1.534378e+10,2.994300e+10,2.087900e+10,8.575000e+10,1.066290e+11,5.695000e+10,3.649800e+11,1.529870e+11,1.763920e+11,AAPL
2,2023-09-30,3.832850e+11,1.691480e+11,1.143010e+11,9.699500e+10,1.258200e+11,1.574423e+10,2.996500e+10,1.580700e+10,9.528100e+10,1.110880e+11,6.214600e+10,3.525830e+11,1.435660e+11,1.453080e+11,AAPL
3,2022-09-30,3.943280e+11,1.707820e+11,1.194370e+11,9.980300e+10,1.305410e+11,1.621596e+10,2.364600e+10,2.111000e+10,9.895900e+10,1.324800e+11,5.067200e+10,3.527550e+11,1.354050e+11,1.539820e+11,AAPL
4,2021-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,AAPL


In [12]:
df_merged = pd.merge(df_prices, df_fundamentals, on=['Ticker', 'Fecha'], how='outer')

# --- LA CORRECCIÓN DEL LOOKAHEAD BIAS ---
# Desplazamos la fecha del reporte 60 días hacia el futuro para simular 
# el momento en que el dato se vuelve realmente público para los inversores.
retraso_dias = 60
df_fundamentals['Fecha'] = df_fundamentals['Fecha'] + pd.DateOffset(days=retraso_dias)
# ----------------------------------------

# 2. Hacer el OUTER merge
df_merged = pd.merge(df_prices, df_fundamentals, on=['Ticker', 'Fecha'], how='outer')

# 3. Ordenar cronológicamente (CRUCIAL para el ffill)
df_merged = df_merged.sort_values(by=['Ticker', 'Fecha'])

# 4. Aplicar Forward Fill
# Identificamos qué columnas vienen de los estados financieros (las que tienen NaN)
cols_financieras = ['Total Revenue', 'Gross Profit', 'Operating Income', 'Net Income', 
                    'EBITDA', 'Basic Average Shares', 'Cash And Cash Equivalents', 'Current Debt', 
                    'Long Term Debt', 'Total Debt', 'Stockholders Equity',
                    'Total Assets', 'Current Assets', 'Current Liabilities'
                    ]

# Aplicamos el ffill() EXCLUSIVAMENTE a esas columnas, agrupando por Ticker
df_merged[cols_financieras] = df_merged.groupby('Ticker')[cols_financieras].ffill()

# 5. Limpiar las filas huérfanas creadas por el outer merge
df_merged_clean = df_merged.dropna(subset=['Close'])

# 6. ELIMINAR LOS NaN INICIALES 
# Filtramos por una columna fundamental crítica. Las filas anteriores al primer
# reporte financiero disponible serán eliminadas de forma limpia.
columna_critica = 'EBITDA' # O 'Net Income', la que uses sí o sí en tus ratios
df_merged_clean = df_merged_clean.dropna(subset=[columna_critica])

# Resetear el índice para que quede estético de 0 a N
df_merged_clean = df_merged_clean.reset_index(drop=True)

# Unir los betas calculados al dataframe limpio final
df_merged_clean = pd.merge(df_merged_clean, df_betas, on=['Ticker', 'Fecha'], how='left')

print(df_merged_clean.head())

       Fecha       Close Ticker  Total Revenue  Gross Profit  \
0 2022-12-05  139.787491   AAPL   3.943280e+11  1.707820e+11   
1 2022-12-12  132.265182   AAPL   3.943280e+11  1.707820e+11   
2 2022-12-19  129.659409   AAPL   3.943280e+11  1.707820e+11   
3 2022-12-26  127.761597   AAPL   3.943280e+11  1.707820e+11   
4 2023-01-02  127.456772   AAPL   3.943280e+11  1.707820e+11   

   Operating Income    Net Income        EBITDA  Basic Average Shares  \
0      1.194370e+11  9.980300e+10  1.305410e+11          1.621596e+10   
1      1.194370e+11  9.980300e+10  1.305410e+11          1.621596e+10   
2      1.194370e+11  9.980300e+10  1.305410e+11          1.621596e+10   
3      1.194370e+11  9.980300e+10  1.305410e+11          1.621596e+10   
4      1.194370e+11  9.980300e+10  1.305410e+11          1.621596e+10   

   Cash And Cash Equivalents  Current Debt  Long Term Debt    Total Debt  \
0               2.364600e+10  2.111000e+10    9.895900e+10  1.324800e+11   
1               2.364600

In [13]:
df_merged_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 18 columns):
 #   Column                     Non-Null Count  Dtype         
---  ------                     --------------  -----         
 0   Fecha                      380 non-null    datetime64[ns]
 1   Close                      380 non-null    float64       
 2   Ticker                     380 non-null    object        
 3   Total Revenue              380 non-null    float64       
 4   Gross Profit               380 non-null    float64       
 5   Operating Income           380 non-null    float64       
 6   Net Income                 380 non-null    float64       
 7   EBITDA                     380 non-null    float64       
 8   Basic Average Shares       380 non-null    float64       
 9   Cash And Cash Equivalents  380 non-null    float64       
 10  Current Debt               380 non-null    float64       
 11  Long Term Debt             380 non-null    float64       
 12  Total De

In [14]:
df_with_metrics = calcular_metricas(df_merged_clean)


In [15]:
# Luego de calcular los ratios, se eliminan las columnas financieras 
df_with_metrics = df_with_metrics.drop(columns=cols_financieras)
df_with_metrics.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Fecha               380 non-null    datetime64[ns]
 1   Close               380 non-null    float64       
 2   Ticker              380 non-null    object        
 3   Beta                314 non-null    float64       
 4   MarketCap           380 non-null    float64       
 5   EnterpriseValue     380 non-null    float64       
 6   PE_Trailing         380 non-null    float64       
 7   EnterpriseToEbitda  380 non-null    float64       
 8   PriceToBook         380 non-null    float64       
 9   operatingMargins    380 non-null    float64       
 10  profitMargins       380 non-null    float64       
 11  returnOnEquity      380 non-null    float64       
 12  ReturnOnAssets      380 non-null    float64       
 13  debtToEquity        380 non-null    float64       

In [16]:
# Unir df_with_metrics con df_tickers
df_final = pd.merge(df_with_metrics, df_tickers, on="Ticker", how="left")
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Fecha               380 non-null    datetime64[ns]
 1   Close               380 non-null    float64       
 2   Ticker              380 non-null    object        
 3   Beta                314 non-null    float64       
 4   MarketCap           380 non-null    float64       
 5   EnterpriseValue     380 non-null    float64       
 6   PE_Trailing         380 non-null    float64       
 7   EnterpriseToEbitda  380 non-null    float64       
 8   PriceToBook         380 non-null    float64       
 9   operatingMargins    380 non-null    float64       
 10  profitMargins       380 non-null    float64       
 11  returnOnEquity      380 non-null    float64       
 12  ReturnOnAssets      380 non-null    float64       
 13  debtToEquity        380 non-null    float64       

In [17]:
# Guardar datos extraidos en fichero raw_data
df_final.to_parquet(f"{data_folder}/raw_data.parquet")